# Importing Libraries and Setting Up Device

This cell imports the necessary libraries for building and training a Named Entity Recognition (NER) model using the Transformers library. It includes libraries for data handling, model training, and evaluation. Additionally, it checks for CUDA availability to determine whether to use GPU or CPU for computations.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer, pipeline
import evaluate

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Loading the Dataset

Here, we load the PII (Personally Identifiable Information) detection dataset from JSON files. The dataset is split into training and test sets, which will be used to train and evaluate the NER model for identifying sensitive information in educational data.

In [ ]:
dataset = load_dataset("json", data_files={
    "train": "/kaggle/input/pii-detection-removal-from-educational-data/train.json",
    "test": "/kaggle/input/pii-detection-removal-from-educational-data/test.json"
})
print(dataset['train'][0])

# Extracting Unique Labels

This code iterates through the training dataset to collect all unique labels present in the data. It then creates bidirectional mappings between labels and their corresponding IDs, which are essential for the token classification task. The number of unique labels is also calculated.

In [ ]:
# Extract all unique labels from the training set
unique_labels = set()
for example in dataset['train']:
    unique_labels.update(example['labels'])
unique_labels = sorted(unique_labels)

print("Labels:", unique_labels)

# Create label-to-ID and ID-to-label mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

num_labels = len(unique_labels)
print("Number of labels:", num_labels)

# Initializing the Tokenizer

We initialize the tokenizer from the Microsoft DeBERTa-v3-base model checkpoint. The `add_prefix_space=True` parameter ensures proper tokenization, especially for subword units, which is crucial for accurate NER performance.

In [ ]:
model_checkpoint = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# Tokenization and Label Alignment Function

This function tokenizes the input text sequences and aligns the corresponding labels with the tokenized outputs. It handles word-to-subword mapping by assigning labels to the first subword token of each word and ignoring subsequent subwords to prevent label duplication. This is a critical step for training accurate NER models.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=1024
    )

    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = dataset['train'].map(tokenize_and_align_labels, batched=True)

# Tokenization for Test Set

A simplified tokenization function specifically for the test dataset. Since the test set doesn't require label alignment (as labels are not provided for evaluation), this function only performs tokenization without the label processing steps.

In [ ]:
def tokenize_test(examples):
    return tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=1024
    )

tokenized_test = dataset['test'].map(tokenize_test, batched=True)

# Creating Tokenized Dataset Dictionary

This cell combines the tokenized training and test datasets into a `DatasetDict` object. This structured format is required by the Transformers library for efficient data loading and batching during model training and evaluation.

In [ ]:
tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "test": tokenized_test
})

# Loading the Pre-trained Model

We load the DeBERTa-v3-base model configured for token classification. The model is initialized with the number of labels, and the label-to-ID and ID-to-label mappings are provided to ensure proper classification output interpretation.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Defining Metrics Computation

This function defines how to compute evaluation metrics for the NER task. It uses the seqeval library to calculate precision, recall, F1 score, and accuracy. The function processes model predictions and true labels, filtering out ignored tokens (-100) for accurate metric calculation.

In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# Setting Up Data Collator

We initialize the `DataCollatorForTokenClassification` which handles dynamic padding of sequences and prepares batches for training. This ensures that all sequences in a batch have the same length, optimizing memory usage and computation efficiency.

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

# Configuring Training Arguments

This cell sets up the training hyperparameters and configuration. It includes settings for learning rate, batch sizes, number of epochs, weight decay, and other training-related parameters. Mixed precision training (FP16) is enabled for faster training on compatible hardware.

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    report_to="none",
    fp16=True
)

# Initializing the Trainer

We create a `Trainer` object that encapsulates the model, training arguments, datasets, tokenizer, data collator, and metrics computation function. This object will handle the entire training and evaluation process.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['train'],  # or split further if needed
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Training the Model

This cell initiates the training process for the NER model. The trainer will iterate through the training dataset for the specified number of epochs, updating the model parameters to minimize the loss and improve performance on the NER task.

In [ ]:
trainer.train()

# Saving the Trained Model and Tokenizer

After training, we save the fine-tuned model and tokenizer to the specified directory. This allows us to reload the model later for inference or further training without having to retrain from scratch.

In [ ]:
model.save_pretrained("./deberta-ner")
tokenizer.save_pretrained("./deberta-ner")

# Creating NER Pipeline and Testing

Finally, we set up a pipeline for Named Entity Recognition using the trained model. This pipeline can be used to extract entities from new text. We test it on a sample text from the dataset to demonstrate its functionality and verify that the model is working correctly.

In [ ]:
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

sample_text = dataset['train'][0]['full_text']
entities = ner_pipeline(sample_text)
print(entities)

# Importing Libraries and Setting Up Device

This cell imports the necessary libraries for building and training a Named Entity Recognition (NER) model using the Transformers library. It includes libraries for data handling, model training, and evaluation. Additionally, it checks for CUDA availability to determine whether to use GPU or CPU for computations.

In [25]:
import numpy as np
import torch
from datasets import load_dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer, pipeline
import evaluate

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


# Loading the Dataset

Here, we load the PII (Personally Identifiable Information) detection dataset from JSON files. The dataset is split into training and test sets, which will be used to train and evaluate the NER model for identifying sensitive information in educational data.

In [26]:
dataset = load_dataset("json", data_files={
    "train": "/kaggle/input/pii-detection-removal-from-educational-data/train.json",
    "test": "/kaggle/input/pii-detection-removal-from-educational-data/test.json"
})
print(dataset['train'][0])

{'document': 7, 'full_text': "Design Thinking for innovation reflexion-Avril 2021-Nathalie Sylla\n\nChallenge & selection\n\nThe tool I use to help all stakeholders finding their way through the complexity of a project is the  mind map.\n\nWhat exactly is a mind map? According to the definition of Buzan T. and Buzan B. (1999, Dessine-moi  l'intelligence. Paris: Les Éditions d'Organisation.), the mind map (or heuristic diagram) is a graphic  representation technique that follows the natural functioning of the mind and allows the brain's  potential to be released. Cf Annex1\n\nThis tool has many advantages:\n\n•  It is accessible to all and does not require significant material investment and can be done  quickly\n\n•  It is scalable\n\n•  It allows categorization and linking of information\n\n•  It can be applied to any type of situation: notetaking, problem solving, analysis, creation of  new ideas\n\n•  It is suitable for all people and is easy to learn\n\n•  It is fun and encourages 

# Extracting Unique Labels

This code iterates through the training dataset to collect all unique labels present in the data. It then creates bidirectional mappings between labels and their corresponding IDs, which are essential for the token classification task. The number of unique labels is also calculated.

In [30]:
# Extract all unique labels from the training set
unique_labels = set()
for example in dataset['train']:
    unique_labels.update(example['labels'])
unique_labels = sorted(unique_labels)

print("Labels:", unique_labels)

# Create label-to-ID and ID-to-label mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

num_labels = len(unique_labels)
print("Number of labels:", num_labels)

Labels: ['B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM', 'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM', 'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL', 'O']
Number of labels: 13


# Initializing the Tokenizer

We initialize the tokenizer from the Microsoft DeBERTa-v3-base model checkpoint. The `add_prefix_space=True` parameter ensures proper tokenization, especially for subword units, which is crucial for accurate NER performance.

In [28]:
model_checkpoint = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


# Tokenization and Label Alignment Function

This function tokenizes the input text sequences and aligns the corresponding labels with the tokenized outputs. It handles word-to-subword mapping by assigning labels to the first subword token of each word and ignoring subsequent subwords to prevent label duplication. This is a critical step for training accurate NER models.

In [31]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=1024
    )

    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = dataset['train'].map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/6807 [00:00<?, ? examples/s]

# Tokenization for Test Set

A simplified tokenization function specifically for the test dataset. Since the test set doesn't require label alignment (as labels are not provided for evaluation), this function only performs tokenization without the label processing steps.

In [32]:
def tokenize_test(examples):
    return tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=1024
    )

tokenized_test = dataset['test'].map(tokenize_test, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

# Creating Tokenized Dataset Dictionary

This cell combines the tokenized training and test datasets into a `DatasetDict` object. This structured format is required by the Transformers library for efficient data loading and batching during model training and evaluation.

In [33]:
tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "test": tokenized_test
})

# Loading the Pre-trained Model

We load the DeBERTa-v3-base model configured for token classification. The model is initialized with the number of labels, and the label-to-ID and ID-to-label mappings are provided to ensure proper classification output interpretation.

In [34]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

# Defining Metrics Computation

This function defines how to compute evaluation metrics for the NER task. It uses the seqeval library to calculate precision, recall, F1 score, and accuracy. The function processes model predictions and true labels, filtering out ignored tokens (-100) for accurate metric calculation.

In [35]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# Setting Up Data Collator

We initialize the `DataCollatorForTokenClassification` which handles dynamic padding of sequences and prepares batches for training. This ensures that all sequences in a batch have the same length, optimizing memory usage and computation efficiency.

In [36]:
data_collator = DataCollatorForTokenClassification(tokenizer)

# Configuring Training Arguments

This cell sets up the training hyperparameters and configuration. It includes settings for learning rate, batch sizes, number of epochs, weight decay, and other training-related parameters. Mixed precision training (FP16) is enabled for faster training on compatible hardware.

In [43]:
training_args = TrainingArguments(
    output_dir="./deberta-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    report_to="none",
    fp16=True
)

# Initializing the Trainer

We create a `Trainer` object that encapsulates the model, training arguments, datasets, tokenizer, data collator, and metrics computation function. This object will handle the entire training and evaluation process.

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['train'],  # or split further if needed
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_36/2890253161.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Training the Model

This cell initiates the training process for the NER model. The trainer will iterate through the training dataset for the specified number of epochs, updating the model parameters to minimize the loss and improve performance on the NER task.

In [46]:
trainer.train()

Step,Training Loss
500,0.031500
1000,0.000500


TrainOutput(global_step=1278, training_loss=0.012542870467024417, metrics={'train_runtime': 3598.565, 'train_samples_per_second': 5.675, 'train_steps_per_second': 0.355, 'total_flos': 9468652387776708.0, 'train_loss': 0.012542870467024417, 'epoch': 3.0})

# Saving the Trained Model and Tokenizer

After training, we save the fine-tuned model and tokenizer to the specified directory. This allows us to reload the model later for inference or further training without having to retrain from scratch.

In [47]:
model.save_pretrained("./deberta-ner")
tokenizer.save_pretrained("./deberta-ner")

('./deberta-ner/tokenizer_config.json',
 './deberta-ner/special_tokens_map.json',
 './deberta-ner/spm.model',
 './deberta-ner/added_tokens.json',
 './deberta-ner/tokenizer.json')

# Creating NER Pipeline and Testing

Finally, we set up a pipeline for Named Entity Recognition using the trained model. This pipeline can be used to extract entities from new text. We test it on a sample text from the training dataset to demonstrate its functionality and verify that the model is working correctly.

In [48]:
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

sample_text = dataset['train'][0]['full_text']
entities = ner_pipeline(sample_text)
print(entities)

Device set to use cuda:0


[{'entity_group': 'NAME_STUDENT', 'score': 0.72591496, 'word': 'N', 'start': 52, 'end': 53}, {'entity_group': 'NAME_STUDENT', 'score': 0.63982373, 'word': 'atha', 'start': 53, 'end': 57}, {'entity_group': 'NAME_STUDENT', 'score': 0.75653076, 'word': 'lie Sylla', 'start': 57, 'end': 66}, {'entity_group': 'NAME_STUDENT', 'score': 0.67757267, 'word': 'N', 'start': 2281, 'end': 2282}, {'entity_group': 'NAME_STUDENT', 'score': 0.47462562, 'word': 'atha', 'start': 2282, 'end': 2286}, {'entity_group': 'NAME_STUDENT', 'score': 0.7095259, 'word': 'lie Sylla', 'start': 2286, 'end': 2295}, {'entity_group': 'NAME_STUDENT', 'score': 0.5000124, 'word': 'atha', 'start': 3649, 'end': 3653}, {'entity_group': 'NAME_STUDENT', 'score': 0.7091531, 'word': 'lie Sylla', 'start': 3653, 'end': 3662}]
